# Block 3: 🛸 Safe Bayesian Optimization via Transductive Active Learning

In [ ]:
# @title: Repo setup {display-mode: "form"}
import os
REPO_OWNER = "eth-fdd-fs26"
REPO_NAME  = "FDD-WE1-public"
BRANCH_NAME = "main"

def _in_colab():
    try:
        import google.colab
        return True
    except Exception:
        return False

if _in_colab():
    token = ""
    try:                                  # private repo (testing): read token from Secrets
        from google.colab import userdata
        token = userdata.get("GITHUB_TOKEN") or ""
    except Exception:                     # public repo (students): no token needed
        token = ""
    auth = f"{token}@" if token else ""
    url = f"https://{auth}github.com/{REPO_OWNER}/{REPO_NAME}.git"
    if not os.path.isdir(REPO_NAME):
        print("Cloning the exercise repo…")
        !git clone -q -b "$BRANCH_NAME" "$url"
    else:                                 # already cloned earlier — refresh to the latest version
        print("Updating the exercise repo to the latest version…")
        !git -C "$REPO_NAME" pull -q "$url" || echo "  (could not pull — using the existing copy)"
    os.chdir(f'/content/{REPO_NAME}')

## 1. Introduction

In [ ]:
import numpy as np

In [ ]:
# @title Global Variables Defining the Environment
# The entire region of interest is [-5, 5] x [-5, 5] with resolution 0.1
EXTENT = [-5, 5, -5, 5] # xmin, xmax, ymin, ymax 
GRID_RES = 101 # Number of points along each axis
GRID_RANGE = np.array(np.linspace(-5, 5, GRID_RES)) # x and y range
X1, X2 = np.meshgrid(GRID_RANGE, GRID_RANGE)
GRID_COORDS = np.stack([X1.ravel(), X2.ravel()], axis=1) # shape: (GRID_RES**2, 2)

In [ ]:
import active_learning as al
al.show_introduction()

Before going any further, let's break down the two core concepts in this coding exercise: Trasductive Active Learning and Safe Bayesian Optimization.

In [ ]:
al.explain_al_and_safebo()

### 1.1 About the Terrain

In [ ]:
al.show_terrain_info()

In [ ]:
# @title Visualize the Region of Interest {display-mode: "form"}
al.visualize_region_of_interest()

Of course, you do not know any of this prior to the exploration mission. Notice that the actual maximizer in the region $f^*(−4.00,−3.60)=6.9617$ is located outside the safety region. If you just run a naïve optimization algorithm on $f^*$, your rover will very likely crash!

### 1.2 Exploration Strategy: Gaussian Processes

Your team has identified a site $X_0 \in S^*$ which is known to be safe. You now ask yourself: 

<div style="text-align: center; font-weight: bold; font-style: italic; color: red;">
    How can I possibly find new points to explore safely based on past observations?
</div>

All of a sudden you recalled from this summer workshop that uncertainty quantification and belief updates can be modelled as Gaussian Processes: as new observations arrive, you update your current estimate and hope to get increasingly accurate predictive models for decision making.

Specifically, you will maintain two GPs $f\sim \text{GP}(\mu,k)$ and $g\sim\text{GP}(\mu',k')$ which model $f^*$ and $g^*$ respectively. As you gather data, the posteriors means should converge to their true values $f^*$ and $g^*$ while the posterior variances shrink. The optimization workflow goes as follows:

In [ ]:
al.safe_bo_workflow()

In [ ]:
al.gp_priors_quiz()

### 1.3 Mapping the Zones

In [ ]:
al.explain_zones_definition()

#### Uncertainty Quantification using Gaussian Processes

In [ ]:
al.explain_confidence_quantification()

#### Formal Definition of Zones and Update Rule

In [ ]:
al.explain_update_rule()

In [ ]:
al.gp_hyperparameters_quiz()

## Part 2: Implmentation

In [ ]:
# @title Import Libraries {display-mode: "form"}
from scipy.linalg import solve 
from active_learning import unflatten, observe, f_star, g_star, bump_function, rbf_kernel

In [ ]:
# @ title A minimal Gaussian Process Class
class GaussianProcess:
    def __init__(self, kernel_func=rbf_kernel, noise_var=0.01, grid_coords=GRID_COORDS, prior_mean=0.0):
        '''Set up the GP with a kernel function, noise variance, grid coordinates, and prior mean.'''
        # Define the grid coordinates for function evaluation
        self.X_grid = grid_coords # Flattened 2D grid coordinates; shape: (GRID_RES**2, 2)
        
        # Define the kernel function and noise variance
        self.kernel_func = kernel_func # Kernel function for the GP
        self.noise_var = noise_var # Variance of the observation noise

        # Initialize the prior mean and covariance for the grid
        self.prior_mean = prior_mean # Scalar; prior mean for the grid
        self.mu_grid = np.full(GRID_RES**2, self.prior_mean) # prior mean; shape: (GRID_RES**2, 2)
        self.K_grid = self.kernel_func(self.X_grid, self.X_grid) # prior covariance; shape: (GRID_RES**2, GRID_RES**2)
        
        # Placeholders for training data
        self.X_train = np.array([]).reshape(-1, 2) # 2D input
        self.y_train = np.array([]) # 1D output

    def update_observations(self, X_train, y_train):
        self.X_train = np.array(X_train).reshape(-1, 2)
        self.y_train = np.array(y_train)
    
    def compute_posterior(self):
        # If no training points have been added, return prior mean and covariance
        if self.X_train.size == 0:
            return self.mu_grid, self.K_grid
        
        # Else, compute the posterior mean and covariance based on the training data
        posterior_mean, posterior_cov = al.compute_posterior_from_observations(self)
        # shape of posterior_mean: (GRID_RES**2,)
        # shape of posterior_cov: (GRID_RES**2, GRID_RES**2)
        
        return posterior_mean, posterior_cov

In [ ]:
# @title 🎯 Your task: Implement the Computation of Zones
def compute_zones(mu_f, cov_f, mu_g, cov_g, beta_f, beta_g):
    '''
    Compute the zones S_n, S_hat, and A_n based on the posterior.
    Parameters:
        mu_f (np.ndarray): Posterior mean of f, shape: (GRID_RES**2,)
        cov_f (np.ndarray): Posterior covariance of f, shape: (GRID_RES**2, GRID_RES**2)
        mu_g (np.ndarray): Posterior mean of g, shape: (GRID_RES**2,)
        cov_g (np.ndarray): Posterior covariance of g, shape: (GRID_RES**2, GRID_RES**2)
        beta_f (float): Beta parameter for f
        beta_g (float): Beta parameter for g
    Returns:
        S_n_mask (np.ndarray): Boolean mask for zone S_n, shape: (GRID_RES, GRID_RES)
        S_hat_n_mask (np.ndarray): Boolean mask for zone S_hat_n, shape: (GRID_RES, GRID_RES)
        A_n_mask (np.ndarray): Boolean mask for zone A_n, shape: (GRID_RES, GRID_RES)
    '''
    
    # 🎯 Compute the standard deviations of f and g at all points in the grid
    # Formula: σ(x) = sqrt(K(x, x)), i.e. the square root of the diagonal of the covariance
    # Hint: np.diag(cov) extracts the variances; then use np.sqrt
    std_f = ... # shape: (GRID_RES**2,)  — from cov_f
    std_g = ... # shape: (GRID_RES**2,)  — from cov_g

    # 🎯 Compute the confidence intervals for f and g
    # Formula (from Section 1.3):
    #   l_f(x) = μ_f(x) − β_f · σ_f(x),   u_f(x) = μ_f(x) + β_f · σ_f(x)
    #   l_g(x) = μ_g(x) − β_g · σ_g(x),   u_g(x) = μ_g(x) + β_g · σ_g(x)
    # Hint: use mu_f / mu_g, beta_f / beta_g, and the stds you just computed (element-wise ±)
    l_f, u_f = ... # shape: (GRID_RES**2,), (GRID_RES**2,)
    l_g, u_g = ... # shape: (GRID_RES**2,), (GRID_RES**2,)

    # Compute the zones as boolean masks (already filled in for you):
    #   S_n     = {x | l_g(x) ≥ 0}                          — safe set
    #   S_hat_n = {x | u_g(x) ≥ 0}                          — optimistic safe set
    #   A_n     = {x ∈ S_hat_n | u_f(x) ≥ max_{x'∈S_n} l_f(x')}  — potential maximizers
    S_n = l_g >= 0.0 # shape: (GRID_RES**2,)
    S_hat_n = u_g >= 0.0 # shape: (GRID_RES**2,)
    max_l_f_S = np.max(l_f[S_n]) if np.any(S_n) else -np.inf
    A_n = S_hat_n & (u_f >= max_l_f_S) # shape: (GRID_RES**2,)

    # Reshape the masks into 2D
    S_n = unflatten(S_n)         # shape: (GRID_RES, GRID_RES)
    S_hat_n = unflatten(S_hat_n) # shape: (GRID_RES, GRID_RES)
    A_n = unflatten(A_n)         # shape: (GRID_RES, GRID_RES)

    return S_n, S_hat_n, A_n

In [ ]:
# @title The function to select the next point based on information gain
def select_next_point_ITL(S_n, A_n, k_f_post, noise_var):
    '''
    Given the precomputed masks S_n, A_n, the posterior covariance of f, and the noise variance, 
    select the next point to query based on information gain.
    Parameters:
        S_n (np.ndarray): Boolean mask for zone S_n, shape: (GRID_RES, GRID_RES)
        A_n (np.ndarray): Boolean mask for zone A_n, shape: (GRID_RES, GRID_RES)
        k_f_post (np.ndarray): Posterior covariance of f, shape: (GRID_RES**2, GRID_RES**2)
        noise_var (float): Noise variance of the observations
    Returns:
        best_point (np.ndarray): Coordinates of the selected point, shape: (2,)
    '''
    # 1. Convert 2D masks back to 1D and collect indices where true
    S_n_mask = S_n.flatten() # shape: (GRID_RES**2,)
    A_n_mask = A_n.flatten() # shape: (GRID_RES**2,)
    S_n_idx = np.where(S_n_mask)[0] # shape: (|S_n|,)
    A_n_idx = np.where(A_n_mask)[0] # shape: (|A_n|,)
    if len(S_n_idx) == 0:
        raise ValueError("No valid points to select: S_n is empty! Decrease your beta_g to explore more points.")
    if len(A_n_idx) == 0:
        raise ValueError("No valid points to select: A_n is empty! Increase your beta_g and/or beta_f to explore more points.")
    
    # 2. Compute the information gain for each point in S_n
    k_xx = np.diag(k_f_post)[S_n_idx]         # shape: (|S_n|,)
    K_AA = k_f_post[np.ix_(A_n_idx, A_n_idx)] # shape: (|A_n|, |A_n|)
    
    # Compute v = K_AA^{-1} @ k_Ax
    k_Ax = k_f_post[np.ix_(A_n_idx, S_n_idx)] # shape: (|A_n|, |S_n|)
    jitter = 1e-5 * np.eye(len(A_n_idx)) # for numerical stability
    v = solve(K_AA + jitter, k_Ax, assume_a='pos') # shape: (|A_n|, |S_n|)    
    assert not np.isnan(v).any(), "Got NaN values in matrix inversion. Increase jitter to enforce positive definiteness"
    
    # Compute the conditional posterior k_tilde_xx = k_xx - diag(k_Ax.T @ K_AA^{-1} @ k_Ax)
    # Notice that diag(A.T @ B) = sum(A * B, axis=0) for matrices A and B
    k_tilde_xx = k_xx - np.sum(k_Ax * v, axis=0) # shape: (|S_n|,)

    # Compute the quantity inside information gain for each point in S_n
    info_gain_proxy = (k_xx + noise_var) / (k_tilde_xx + noise_var) # shape: (|S_n|,)
    
    # 3. Find the point with the highest information gain
    best = np.argmax(info_gain_proxy) # index in S_n_idx
    best_idx = S_n_idx[best] # index in the flattened grid
    best_point = GRID_COORDS[best_idx] # coordinates of the selected point
    return best_point

In [ ]:
# @title 🎯 Mission class to handle optimization progress and UI
class Mission:
    def __init__(self, f_mu_prior: float, g_mu_prior: float, noise_var_f: float, noise_var_g: float, 
                 beta_f: float, beta_g: float, x_init: np.ndarray):
        '''The mission class encapsulates the active learning process for the rover.
        Parameters:
            - f_mu_prior, g_mu_prior: Prior mean for function f and g
            - noise_var_f, noise_var_g: Noise variance for function f and g
            - beta_f, beta_g: Confidence parameter for function f and g
            - x_init: Initial point for active learning
        '''
        # Initialize the UI components for the mission visualization
        al.initialize_mission_ui(self)

        # Initialize GPs for f and g
        self.f = GaussianProcess(grid_coords=GRID_COORDS, prior_mean=f_mu_prior, noise_var=noise_var_f)
        self.g = GaussianProcess(grid_coords=GRID_COORDS, prior_mean=g_mu_prior, noise_var=noise_var_g)        
        self.g.mu_grid = bump_function(center=x_init, radius=1, inf_val=self.g.prior_mean, max_val=1, 
                                       X=self.g.X_grid) # Set prior indicating g is safe around x_init
        
        # Initialize the mission state variables
        self.x_init = x_init
        self.beta_f = beta_f
        self.beta_g = beta_g
        self.reset_data_state()


    def reset_data_state(self):
        # Clear all historical data and start fresh with the initial point
        self.round = 0              # current round of drilling
        self.x_next = self.x_init   # next point to query
        self.X_history = []     # history of queried points
        self.y_history = []     # history of observed f values
        self.z_history = []     # history of observed g values
        al.reset_ui_data(self) # Reset the UI data for the mission
        
        # Initialize posteriors to priors
        self.mu_f_post, self.cov_f_post = self.f.compute_posterior()
        self.mu_g_post, self.cov_g_post = self.g.compute_posterior()
        
        # Compute initial boolean masks for zones
        self.S_n, self.S_hat_n, self.A_n = compute_zones(self.f.mu_grid, self.f.K_grid, \
                    self.g.mu_grid, self.g.K_grid, beta_f=self.beta_f, beta_g=self.beta_g)

    def conduct_drilling_round(self):
        # 🎯 Complete the method that carries out the worflow for each round
        # Step 1: Observe function values at the selected point and add to history
        observed_f = float(observe(f_star, self.x_next[None,:], self.f.noise_var)[0])
        observed_g = float(observe(g_star, self.x_next[None,:], self.g.noise_var)[0])
        self.X_history.append(self.x_next)
        self.y_history.append(observed_f)
        self.z_history.append(observed_g)
        al.record_confidence_intervals(self) # Record the current CIs for f and g

        # Step 2: Check if constraint is violated at the observed point
        al.check_constraint_violation(self) 

        # Step 3: Update the GP models with the latest observations    
        self.f.update_observations(np.array(self.X_history), np.array(self.y_history))
        self.g.update_observations(np.array(self.X_history), np.array(self.z_history))
        
        # 🎯 Step 4: Compute the posterior means and covariances for both f and g
        # Hint: call compute_posterior() on self.f and self.g
        #   (returns a tuple: posterior_mean, posterior_cov)
        self.mu_f_post, self.cov_f_post = ...
        self.mu_g_post, self.cov_g_post = ...

        # 🎯 Step 5: Compute the zones based on the posterior means and covariances
        # Hint: call compute_zones(...) with the posteriors from Step 4 and the betas
        #   Signature: compute_zones(mu_f, cov_f, mu_g, cov_g, beta_f, beta_g)
        #   Use: self.mu_f_post, self.cov_f_post, self.mu_g_post, self.cov_g_post,
        #        self.beta_f, self.beta_g
        self.S_n, self.S_hat_n, self.A_n = ...

        # 🎯 Step 6: Find the next point to query based on the zones and posterior covariance of f
        # Hint: call select_next_point_ITL(S_n, A_n, k_f_post, noise_var)
        #   Use: self.S_n, self.A_n, self.cov_f_post, and self.f.noise_var
        self.x_next = ...
    
        # Step 7: Update the current best point based on posteriors
        al.update_best_point(self) # Check the point with the highest f^* value in S_n
        
        # Step 8: Increment the round counter and display progress
        self.round += 1
        al.render_mission_dashboard(self)        

## Part 3: Running the Mission
Run the cell below and you will see an interactive mission dashboard. Click "Drill Next Step" to conduct the next drilling and "Reset Mission" to start over. You can see the zones and beliefs for $f$ in each round as well as observation points on the map, and the log tells you how well your rover is doing. 

Optionally, change the parameters in the next cell to see what effects they have.

In [ ]:
# @title Create and run Mission 
X_INIT = np.array([-1.0, -1.0]) # Initial point for active learning
F_MU_PRIOR = 8.0 # Prior mean for f
G_MU_PRIOR = -3.0 # Prior mean for g
BETA_F = 2.0 # Confidence parameter for f
BETA_G = 0.2 # Confidence parameter for g
NOISE_VAR_F = 0.1 # Noise variance for f
NOISE_VAR_G = 0.01 # Noise variance for g
mission = Mission(f_mu_prior=F_MU_PRIOR, g_mu_prior=G_MU_PRIOR, noise_var_f=NOISE_VAR_F, noise_var_g=NOISE_VAR_G, 
                  beta_f=BETA_F, beta_g=BETA_G, x_init=X_INIT)
al.display_mission_ui(mission)